# 00 语音合成 TTS：ModelScope 小模型 + 内部结构拆解

这份 notebook 重新从一个更适合教学的小型 TTS 模型开始：先用 ModelScope 下载 `microsoft/speecht5_tts`，再用 Transformers 直接加载 `SpeechT5ForTextToSpeech`，这样我们不仅能合成一段 WAV，还能真正打开模型内部结构看清楚：

- 文本如何变成 `input_ids`
- TTS 模型里 encoder、decoder、postnet 分别做什么
- vocoder 为什么是单独一段模型
- 怎么统计参数量、查看子模块、抓取中间 shape

这里不再默认使用 Sambert + HiFiGAN 的 ModelScope pipeline，因为它在 Python 3.11/3.12 环境里容易卡 `ttsfrd` 兼容问题，而且 pipeline 封装太厚，不利于查看内部结构。

## 1. 安装依赖

在仓库根目录运行：

```bash
pip install -r requirements.txt
pip install -r speech/requirements-speech.txt
```

这个 notebook 用 ModelScope 负责下载模型文件，用 Transformers 负责加载和拆解结构。默认模型是：

```text
SMALL_TTS_MODEL_ID=microsoft/speecht5_tts
VOCODER_MODEL_ID=microsoft/speecht5_hifigan
```

默认 SpeechT5 TTS 模型更适合英文示例；如果你换成中文 TTS 模型，再把后面的 `SYNTH_TEXT` 改成中文。

In [ ]:
from pathlib import Path
import json
import os
import time
from collections import OrderedDict

import soundfile as sf
import torch
import torch.nn.functional as F

OUTPUT_DIR = Path("speech/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SYNTH_TEXT = "Text to speech models turn written words into an audio waveform."

# 优先使用环境变量，方便你在 ModelScope 页面复制模型 ID 后直接替换。
SMALL_TTS_MODEL_ID = os.getenv("SMALL_TTS_MODEL_ID", "microsoft/speecht5_tts")
VOCODER_MODEL_ID = os.getenv("VOCODER_MODEL_ID", "microsoft/speecht5_hifigan")

# 有些 ModelScope 镜像会带组织名前缀。默认先试主 ID，再试候选 ID。
SMALL_TTS_MODEL_CANDIDATES = [
    SMALL_TTS_MODEL_ID,
    "AI-ModelScope/speecht5_tts",
]
VOCODER_MODEL_CANDIDATES = [
    VOCODER_MODEL_ID,
    "AI-ModelScope/speecht5_hifigan",
]

RUN_GENERATION = True
RUN_SHAPE_HOOK_DEMO = True

print("torch", torch.__version__)
print("small tts candidates:", SMALL_TTS_MODEL_CANDIDATES)
print("vocoder candidates:", VOCODER_MODEL_CANDIDATES)
print("text:", SYNTH_TEXT)

## 2. 下载模型：ModelScope 只负责拿到本地目录

`snapshot_download()` 会把模型权重和配置缓存到本机。后面我们不直接调用黑盒 `pipeline`，而是把这个本地目录交给 Transformers：

```text
ModelScope model id
-> snapshot_download
-> local model_dir
-> SpeechT5Processor / SpeechT5ForTextToSpeech / SpeechT5HifiGan
```

In [ ]:
def unique_items(items):
    seen = OrderedDict()
    for item in items:
        if item and item not in seen:
            seen[item] = None
    return list(seen.keys())


def pick_torch_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def download_first_available(model_ids, purpose):
    from modelscope import snapshot_download

    errors = []
    for model_id in unique_items(model_ids):
        try:
            print(f"Downloading {purpose}: {model_id}")
            model_dir = snapshot_download(model_id)
            print(f"{purpose} cache: {model_dir}")
            return model_id, model_dir
        except Exception as exc:
            errors.append(f"{model_id}: {type(exc).__name__}: {exc}")
            print(f"Skip {model_id}: {type(exc).__name__}: {exc}")

    message = "\n".join(errors)
    raise RuntimeError(
        f"Could not download {purpose} from ModelScope. "
        f"Set an available ModelScope model id with an environment variable.\n{message}"
    )


selected_tts_model_id, tts_model_dir = download_first_available(
    SMALL_TTS_MODEL_CANDIDATES,
    "SpeechT5 TTS model",
)
selected_vocoder_model_id, vocoder_model_dir = download_first_available(
    VOCODER_MODEL_CANDIDATES,
    "SpeechT5 HiFiGAN vocoder",
)

selected_tts_model_id, selected_vocoder_model_id

## 3. 加载模型：不要只看 pipeline，要拿到真实模块

TTS 模型可以拆成两段：

| 模块 | 作用 | 在本 notebook 里的对象 |
| --- | --- | --- |
| processor / tokenizer | 把文字转成 token id | `SpeechT5Processor` |
| acoustic model | 从文本和说话人条件生成声学表示 | `SpeechT5ForTextToSpeech` |
| vocoder | 把声学表示还原为 waveform | `SpeechT5HifiGan` |

这样加载后，`print(model)`、`model.named_modules()`、`model.config` 都可以直接查看。

In [ ]:
from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, SpeechT5Processor

device = pick_torch_device()
print("device:", device)

processor = SpeechT5Processor.from_pretrained(tts_model_dir)
model = SpeechT5ForTextToSpeech.from_pretrained(tts_model_dir).to(device)
vocoder = SpeechT5HifiGan.from_pretrained(vocoder_model_dir).to(device)

model.eval()
vocoder.eval()

print("processor:", type(processor).__name__)
print("model:", type(model).__name__)
print("vocoder:", type(vocoder).__name__)

## 4. 先看 config：模型有哪些关键超参数

查看模型结构时不要一上来就被 `print(model)` 淹没。先看 config 里的关键字段：隐藏层维度、层数、注意力头数、speaker embedding 维度等。这些字段通常决定模型的计算量和参数规模。

In [ ]:
def show_config(config, keys):
    config_dict = config.to_dict()
    for key in keys:
        if key in config_dict:
            print(f"{key}: {config_dict[key]}")


interesting_keys = [
    "model_type",
    "hidden_size",
    "encoder_layers",
    "decoder_layers",
    "encoder_attention_heads",
    "decoder_attention_heads",
    "encoder_ffn_dim",
    "decoder_ffn_dim",
    "speaker_embedding_dim",
    "reduction_factor",
    "num_mel_bins",
    "max_text_positions",
    "max_speech_positions",
]

show_config(model.config, interesting_keys)
print("\nall config keys:")
print(sorted(model.config.to_dict().keys()))

## 5. 看文字输入：TTS 的输入不是音频，而是 token id

对 TTS 来说，输入的第一步仍然是 NLP：文本先经过 tokenizer，变成模型能处理的 `input_ids`。后续声学模型才会把这些文本表示转成语音相关的中间表示。

In [ ]:
inputs = processor(text=SYNTH_TEXT, return_tensors="pt")
input_ids = inputs["input_ids"]

print("input_ids shape:", tuple(input_ids.shape))
print("input_ids:", input_ids[0].tolist())

tokenizer = getattr(processor, "tokenizer", None)
if tokenizer is not None:
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    print("tokens:", tokens)

## 6. 统计参数量：先看大块，再看细节

一个实用习惯：先统计总参数量，再按一级子模块拆开。这样你能马上知道参数主要在 encoder、decoder、postnet 还是其他部分。

In [ ]:
def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())


def format_params(count):
    if count >= 1_000_000_000:
        return f"{count / 1_000_000_000:.2f}B"
    if count >= 1_000_000:
        return f"{count / 1_000_000:.2f}M"
    if count >= 1_000:
        return f"{count / 1_000:.2f}K"
    return str(count)


print("model params:", format_params(count_parameters(model)))
print("vocoder params:", format_params(count_parameters(vocoder)))
print("\nmodel top-level children:")
for name, child in model.named_children():
    print(f"{name:<32} {type(child).__name__:<36} {format_params(count_parameters(child))}")

## 7. 打印结构树：看 encoder、decoder、postnet 在哪里

`print(model)` 会非常长。下面这个函数只打印到指定深度，适合先建立地图。你可以把 `max_depth` 调大继续展开。

In [ ]:
def module_tree(module, max_depth=2):
    lines = []

    def visit(prefix, current, depth):
        if depth > max_depth:
            return
        for name, child in current.named_children():
            full_name = f"{prefix}.{name}" if prefix else name
            lines.append(
                f"{'  ' * depth}{full_name}: {type(child).__name__} "
                f"({format_params(count_parameters(child))})"
            )
            visit(full_name, child, depth + 1)

    visit("", module, 0)
    return lines


print("\n".join(module_tree(model, max_depth=2)))

## 8. 按关键词找模块：定位注意力层、encoder、decoder

真正读源码或调模型时，经常用 `named_modules()` 按关键词找模块。比如你想看 attention 在哪里，或者想给 encoder / decoder 挂 hook。

In [ ]:
def find_modules_by_keyword(module, keywords, limit=60):
    keywords = [keyword.lower() for keyword in keywords]
    rows = []
    for name, child in module.named_modules():
        lowered = name.lower()
        if name and any(keyword in lowered for keyword in keywords):
            rows.append((name, type(child).__name__, format_params(count_parameters(child))))
        if len(rows) >= limit:
            break
    return rows


for name, cls_name, params in find_modules_by_keyword(
    model,
    ["encoder", "decoder", "attention", "postnet", "prenet"],
    limit=80,
):
    print(f"{name:<70} {cls_name:<34} {params}")

## 9. 合成一段语音：文本 + 说话人 embedding + vocoder

SpeechT5 这类 TTS 模型需要一个 speaker embedding 来指定“谁在说”。真实项目里通常会使用说话人向量或参考音频提取的 embedding。为了让教程不依赖额外数据集，这里用固定随机种子生成一个归一化 speaker embedding；它适合跑通链路和观察结构，不代表最佳音色。

生成链路是：

```text
text -> input_ids
input_ids + speaker_embedding -> acoustic representation
acoustic representation + vocoder -> waveform
waveform -> WAV file
```

In [ ]:
def make_demo_speaker_embedding(model):
    speaker_dim = getattr(model.config, "speaker_embedding_dim", 512)
    generator = torch.Generator(device="cpu").manual_seed(2026)
    embedding = torch.randn((1, speaker_dim), generator=generator)
    embedding = F.normalize(embedding, dim=-1)
    return embedding.to(device)


def save_wav(path, waveform, sample_rate, elapsed_seconds):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, waveform, sample_rate)
    info = sf.info(path)
    duration_seconds = info.frames / info.samplerate if info.samplerate else 0.0
    rtf = elapsed_seconds / duration_seconds if duration_seconds else float("inf")
    print(f"saved: {path}")
    print(
        f"sample_rate={info.samplerate}, duration={duration_seconds:.2f}s, "
        f"elapsed={elapsed_seconds:.2f}s, rtf={rtf:.2f}"
    )
    return {
        "path": str(path),
        "sample_rate": info.samplerate,
        "duration_seconds": duration_seconds,
        "elapsed_seconds": elapsed_seconds,
        "rtf": rtf,
    }


if RUN_GENERATION:
    speaker_embeddings = make_demo_speaker_embedding(model)
    model_inputs = {name: value.to(device) for name, value in inputs.items()}

    start = time.perf_counter()
    with torch.inference_mode():
        speech = model.generate_speech(
            model_inputs["input_ids"],
            speaker_embeddings,
            vocoder=vocoder,
        )
    elapsed = time.perf_counter() - start

    speech_array = speech.detach().cpu().numpy()
    speech_metrics = save_wav(
        OUTPUT_DIR / "speecht5_modelscope_structure_demo.wav",
        speech_array,
        sample_rate=16_000,
        elapsed_seconds=elapsed,
    )
    speech_metrics
else:
    print("Generation skipped. Set RUN_GENERATION = True to synthesize audio.")

## 10. 用 forward hook 看中间 shape

`forward hook` 可以在模块运行时记录输入输出 shape。TTS 的 `generate_speech()` 内部会多步自回归生成，hook 输出会比较多，所以这里默认只挂几个关键模块：encoder、decoder、postnet。你可以修改 `HOOK_MODULE_NAMES` 继续观察更细的 attention 层。

In [ ]:
def tensor_shape(value):
    if isinstance(value, torch.Tensor):
        return tuple(value.shape)
    if isinstance(value, (list, tuple)):
        return [tensor_shape(item) for item in value[:3]]
    if isinstance(value, dict):
        return {key: tensor_shape(item) for key, item in list(value.items())[:5]}
    return type(value).__name__


def attach_shape_hooks(module, module_names):
    modules = dict(module.named_modules())
    records = []
    handles = []

    for name in module_names:
        if name not in modules:
            print(f"missing module for hook: {name}")
            continue

        def make_hook(module_name):
            def hook(_module, inputs, output):
                records.append(
                    {
                        "module": module_name,
                        "input": tensor_shape(inputs),
                        "output": tensor_shape(output),
                    }
                )
            return hook

        handles.append(modules[name].register_forward_hook(make_hook(name)))
        print(f"hooked: {name} -> {type(modules[name]).__name__}")

    return handles, records


HOOK_MODULE_NAMES = [
    "speecht5.encoder",
    "speecht5.decoder",
    "speech_decoder_postnet",
]

if RUN_SHAPE_HOOK_DEMO and RUN_GENERATION:
    handles, shape_records = attach_shape_hooks(model, HOOK_MODULE_NAMES)
    try:
        with torch.inference_mode():
            _ = model.generate_speech(
                model_inputs["input_ids"],
                speaker_embeddings,
                vocoder=vocoder,
            )
    finally:
        for handle in handles:
            handle.remove()

    print("\nshape records, first 12:")
    for record in shape_records[:12]:
        print(json.dumps(record, ensure_ascii=False))
else:
    print("Shape hook demo skipped. Set RUN_SHAPE_HOOK_DEMO = True and RUN_GENERATION = True.")

## 11. 读结构时应该怎么讲

面试或项目复盘里，可以这样描述这个小模型：

> 这个 TTS demo 不是只调用一个黑盒 pipeline，而是把语音合成拆成三层：processor 把文本变成 token id；SpeechT5 声学模型用 encoder-decoder 结构把文本和 speaker embedding 转成声学表示；HiFiGAN vocoder 再把声学表示还原成 waveform。部署时除了音质，还要看参数量、采样率、生成耗时、RTF、说话人条件来源，以及长文本切分策略。

继续深入可以做三件事：

1. 把 speaker embedding 换成真实说话人向量，对比音色稳定性。
2. 给 attention 层挂 hook，观察文本 token 和生成帧之间的对齐。
3. 把同一段文本分别交给小模型和 VoxCPM2，对比采样率、RTF、音质和控制能力。

## 12. 参考链接

- ModelScope: https://modelscope.cn
- SpeechT5 model id: `microsoft/speecht5_tts`
- SpeechT5 vocoder id: `microsoft/speecht5_hifigan`
- Transformers SpeechT5 classes: `SpeechT5Processor`, `SpeechT5ForTextToSpeech`, `SpeechT5HifiGan`